<a href="https://colab.research.google.com/github/galipaydin/pes517/blob/main/notebooks/lab10_tracking_and_event_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# PES 517 · Lab 10 — Tracking and event data
**Artificial Intelligence Applications in Sport · Week 11**

⏱️ About 90 minutes · 🧪 This notebook creates its own data — nothing can fail to download

> **First:** `File → Save a copy in Drive`. Cells marked ✏️ **EDIT ME** are the ones you change.


### What you will produce
Two figures a coach could use: a movement profile built from player tracking, and a shot map built from real
professional event data — plus a demonstration of the failure that quietly corrupts every tracking statistic.

### Three words that are not synonyms
| | |
|---|---|
| **Detection** | Find the players in *this frame*. No memory of the previous frame. |
| **Tracking** | Keep the same identity across frames. This is where things go wrong. |
| **Re-identification** | Recover an identity after the player was hidden or left the frame. |

And the step that makes any of it meaningful: **homography** — the transform from pixels to metres on the pitch.
Without it, a tracking output is decoration.


---
## Block 1 — Setup


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, Arc
RNG = np.random.default_rng(11)
PITCH_L, PITCH_W = 105.0, 68.0   # metres, standard
print('Setup complete.')


---
## Block 2 — A pitch to draw on

Fifteen lines. You do not need a library for this, and drawing it yourself makes the coordinate system explicit —
which is the thing students get wrong when they use somebody else's plotting package.


In [ ]:
def pitch(ax=None, line='#8A8078', bg='none'):
    if ax is None: _, ax = plt.subplots(figsize=(9, 6))
    ax.add_patch(Rectangle((0,0), PITCH_L, PITCH_W, fill=(bg!='none'), fc=bg, ec=line, lw=1.2))
    ax.plot([PITCH_L/2, PITCH_L/2], [0, PITCH_W], color=line, lw=1.2)
    ax.add_patch(Circle((PITCH_L/2, PITCH_W/2), 9.15, fill=False, ec=line, lw=1.2))
    for x0, sgn in [(0, 1), (PITCH_L, -1)]:
        ax.add_patch(Rectangle((x0, PITCH_W/2-20.16), sgn*16.5, 40.32, fill=False, ec=line, lw=1.2))
        ax.add_patch(Rectangle((x0, PITCH_W/2-9.16),  sgn*5.5,  18.32, fill=False, ec=line, lw=1.2))
        ax.plot(x0 + sgn*11, PITCH_W/2, 'o', ms=2, color=line)
    ax.set_xlim(-3, PITCH_L+3); ax.set_ylim(-3, PITCH_W+3)
    ax.set_aspect('equal'); ax.axis('off')
    return ax

pitch(); plt.title('105 × 68 m, origin bottom-left'); plt.tight_layout(); plt.show()


---
# PART A — Tracking

## Block 3 — Player trajectories

Two minutes of play at 10 Hz for one team of eleven — the sampling rate a real optical or LPS system delivers.
Each player oscillates around a positional home, drifts with the ball, and occasionally sprints.


In [ ]:
FS_TRACK, MINUTES = 10, 2
n = FS_TRACK * 60 * MINUTES
t = np.arange(n) / FS_TRACK

HOMES = {'GK':(6,34), 'RB':(30,10), 'RCB':(24,26), 'LCB':(24,42), 'LB':(30,58),
         'RCM':(50,22), 'CM':(46,34), 'LCM':(50,46), 'RW':(74,12), 'ST':(82,34), 'LW':(74,56)}

ball_x = PITCH_L/2 + 26*np.sin(2*np.pi*t/38) + 9*np.sin(2*np.pi*t/7.3)
ball_y = PITCH_W/2 + 17*np.sin(2*np.pi*t/23 + 1.1)

trk = []
for pid, (name, (hx, hy)) in enumerate(HOMES.items()):
    pull = 0.06 if name == 'GK' else RNG.uniform(.26, .46)
    x = hx + pull*(ball_x - PITCH_L/2) + RNG.normal(0, 1.6, n).cumsum()*.05
    y = hy + pull*(ball_y - PITCH_W/2) + RNG.normal(0, 1.6, n).cumsum()*.05
    for _ in range(RNG.integers(2, 6)):                      # a few sprints
        s = RNG.integers(0, n-90)
        x[s:s+90] += np.linspace(0, RNG.uniform(-16, 16), 90)
    trk.append(pd.DataFrame({'t': t, 'player': name, 'shirt': pid+1,
                             'x': np.clip(x, 0, PITCH_L), 'y': np.clip(y, 0, PITCH_W)}))

trk = pd.concat(trk, ignore_index=True)
print(f'{len(trk):,} rows · {trk.player.nunique()} players · {FS_TRACK} Hz · {MINUTES} min')
trk.head()


---
## Block 4 — From coordinates to a movement profile

Speed is the first derivative of position. Everything a GPS vendor sells you — distance, high-speed running,
accelerations — comes from these two columns and a set of thresholds **the vendor chose**.


In [ ]:
HIGH_SPEED_MS = 5.5      # ✏️ EDIT ME — the 'high speed' threshold. Vendors use 5.0-5.8 m/s. Try both.

prof = []
for name, g in trk.groupby('player', sort=False):
    dx, dy = np.diff(g.x), np.diff(g.y)
    step   = np.hypot(dx, dy)
    speed  = step * FS_TRACK
    speed  = pd.Series(speed).rolling(5, center=True, min_periods=1).mean().values   # smooth
    prof.append(dict(player=name,
                     distance_m      = step.sum(),
                     high_speed_m    = step[speed > HIGH_SPEED_MS].sum(),
                     peak_speed_ms   = speed.max(),
                     mean_speed_ms   = speed.mean()))

prof = pd.DataFrame(prof).sort_values('distance_m', ascending=False)
prof[['distance_m','high_speed_m','peak_speed_ms','mean_speed_ms']] = \
    prof[['distance_m','high_speed_m','peak_speed_ms','mean_speed_ms']].round(1)
prof


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = pitch(axes[0])
for name, g in trk.groupby('player', sort=False):
    ax.plot(g.x, g.y, lw=.7, alpha=.75)
axes[0].set_title('All eleven trajectories, 2 minutes')

ax = pitch(axes[1])
st = trk[trk.player == 'ST']
h = axes[1].hist2d(st.x, st.y, bins=[21, 14], range=[[0,PITCH_L],[0,PITCH_W]], cmap='Reds', alpha=.85)
axes[1].set_title('Striker — occupancy heat map')
pitch(axes[1])
plt.tight_layout(); plt.show()


---
## Block 5 — Break it 💥 The identity swap

Two players in the same kit cross paths. The tracker, which matches detections between frames by proximity,
assigns each the other's identity from that moment on. Nothing errors. Nothing looks wrong on the video.
Every per-player statistic downstream is now false.


In [ ]:
a, b = 'RCM', 'LCM'
A = trk[trk.player == a].reset_index(drop=True).copy()
B = trk[trk.player == b].reset_index(drop=True).copy()

# force them to cross at t = 60 s, then swap identities from there on
cross = 60 * FS_TRACK
blend = np.clip((np.arange(len(A)) - (cross-40)) / 40, 0, 1)
mid_x, mid_y = (A.x + B.x)/2, (A.y + B.y)/2
A['x'], A['y'] = A.x*(1-blend*.5) + mid_x*blend*.5, A.y*(1-blend*.5) + mid_y*blend*.5
B['x'], B['y'] = B.x*(1-blend*.5) + mid_x*blend*.5, B.y*(1-blend*.5) + mid_y*blend*.5

sw_A = pd.concat([A.iloc[:cross], B.iloc[cross:]], ignore_index=True)
sw_B = pd.concat([B.iloc[:cross], A.iloc[cross:]], ignore_index=True)

def dist(df): return np.hypot(np.diff(df.x), np.diff(df.y)).sum()

cmp = pd.DataFrame({
    'player':        [a, b],
    'true_m':        [round(dist(A),1), round(dist(B),1)],
    'after_swap_m':  [round(dist(sw_A),1), round(dist(sw_B),1)],
})
cmp['error_m'] = (cmp.after_swap_m - cmp.true_m).round(1)
cmp['error_%'] = (100*cmp.error_m/cmp.true_m).round(1)

fig, ax = plt.subplots(figsize=(9, 5.6)); pitch(ax)
ax.plot(A.x, A.y, color='#1B6470', lw=1.1, label=f'{a} (true)')
ax.plot(B.x, B.y, color='#9A5714', lw=1.1, label=f'{b} (true)')
ax.plot(sw_A.x[cross:], sw_A.y[cross:], color='#8A1C2B', lw=2, ls=':', label=f'{a} after the swap')
ax.plot(A.x[cross], A.y[cross], 'X', ms=13, color='#8A1C2B')
ax.legend(loc='upper left', frameon=False, fontsize=9)
ax.set_title('One identity swap at t = 60 s'); plt.tight_layout(); plt.show()

print(cmp.to_string(index=False))
print('\nNobody would spot this on the video, and the report still prints two plausible numbers.')


---
# PART B — Event data

## Block 6 — StatsBomb open data

StatsBomb release genuine professional event data for free. Try the live path first; the cell falls back to a
built-in match if there is no network.


In [ ]:
COMPETITION, SEASON = 43, 3      # ✏️ EDIT ME — 43/3 is the 2018 men's World Cup

def normalise(raw):
    """StatsBomb returns locations as [x, y] lists. Flatten them into plain columns."""
    out = pd.DataFrame(index=raw.index)
    out['type'] = raw['type']
    out['team'] = raw['team']
    loc = raw['location'].apply(lambda v: v if isinstance(v, (list, tuple)) else [np.nan, np.nan])
    out['x'] = [v[0] for v in loc]
    out['y'] = [v[1] for v in loc]
    end = raw.get('pass_end_location', pd.Series([None]*len(raw), index=raw.index))
    end = end.apply(lambda v: v if isinstance(v, (list, tuple)) else [np.nan, np.nan])
    out['end_x'] = [v[0] for v in end]
    out['end_y'] = [v[1] for v in end]
    # StatsBomb records pass_outcome only when the pass FAILED — a blank means it was completed
    po = raw.get('pass_outcome', pd.Series([np.nan]*len(raw), index=raw.index))
    so = raw.get('shot_outcome', pd.Series([np.nan]*len(raw), index=raw.index))
    out['outcome'] = np.where(raw['type'].eq('Shot'), so.astype(str),
                              np.where(po.isna(), 'complete', 'incomplete'))
    out['shot_statsbomb_xg'] = raw.get('shot_statsbomb_xg', np.nan)
    return out

events, source = None, 'fallback'
try:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'statsbombpy==1.14.0'], timeout=240)
    from statsbombpy import sb
    matches  = sb.matches(competition_id=COMPETITION, season_id=SEASON)
    match_id = int(matches.sort_values('match_date').iloc[-1].match_id)   # the final
    events   = normalise(sb.events(match_id=match_id))
    source   = f'StatsBomb match {match_id}'
    print(f'Loaded {len(events)} real events from {source}')
except Exception as e:
    print(f'Live data unavailable ({type(e).__name__}) — using the built-in match instead.')

if events is None:
    n_p, n_s = 780, 26
    passes = pd.DataFrame({
        'type': 'Pass', 'team': RNG.choice(['Home','Away'], n_p, p=[.56,.44]),
        'x': RNG.uniform(2, 118, n_p), 'y': RNG.uniform(2, 78, n_p),
        'end_x': np.nan, 'end_y': np.nan,
        'outcome': RNG.choice(['complete','incomplete'], n_p, p=[.83,.17]),
        'shot_statsbomb_xg': np.nan})
    passes['end_x'] = np.clip(passes.x + RNG.normal(9, 14, n_p), 0, 120)
    passes['end_y'] = np.clip(passes.y + RNG.normal(0, 13, n_p), 0, 80)
    sx = np.clip(RNG.normal(102, 9, n_s), 78, 119)
    sy = np.clip(RNG.normal(40, 9, n_s), 12, 68)
    dd = np.hypot(120-sx, 40-sy)
    shots = pd.DataFrame({
        'type':'Shot', 'team': RNG.choice(['Home','Away'], n_s, p=[.58,.42]),
        'x': sx, 'y': sy, 'end_x': 120.0, 'end_y': 40.0, 'outcome': None,
        'shot_statsbomb_xg': np.round(np.clip(0.85*np.exp(-dd/7.5) + RNG.normal(0,.02,n_s), .01, .95), 3)})
    shots['outcome'] = np.where(RNG.random(n_s) < shots.shot_statsbomb_xg, 'Goal', 'Off T')
    events = pd.concat([passes, shots], ignore_index=True)

print(f'\nsource: {source}   ·   {len(events)} events · columns: {list(events.columns)}')


> **StatsBomb pitch coordinates are 120 × 80, not 105 × 68.** Every provider uses its own convention, and
> mixing two providers' coordinates without converting is one of the commonest errors in applied analysis.


---
## ✏️ EDIT ME — pass map and shot map


In [ ]:
TEAM = sorted(events.team.dropna().unique())[0]   # ✏️ EDIT ME — pick either team
SB_L, SB_W = 120.0, 80.0
to_m = lambda x, y: (x*PITCH_L/SB_L, y*PITCH_W/SB_W)

ps = events[(events.type == 'Pass') & (events.team == TEAM)].dropna(subset=['x','y','end_x','end_y'])
sh = events[(events.type == 'Shot') & (events.team == TEAM)].dropna(subset=['x','y'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5.4))

ax = pitch(axes[0])
for _, r in ps.iterrows():
    x0, y0 = to_m(r.x, r.y); x1, y1 = to_m(r.end_x, r.end_y)
    ok = str(r['outcome']).lower() == 'complete'
    ax.annotate('', xy=(x1,y1), xytext=(x0,y0),
                arrowprops=dict(arrowstyle='->', lw=.45,
                                color='#1B6470' if ok else '#C9B7B4', alpha=.75))
axes[0].set_title(f'{TEAM} — {len(ps)} passes')

ax = pitch(axes[1])
xg = sh.shot_statsbomb_xg.fillna(0.05)
goal = sh.outcome.astype(str).str.contains('Goal')
for (_, r), g, v in zip(sh.iterrows(), goal, xg):
    x, y = to_m(r.x, r.y)
    ax.scatter(x, y, s=60 + 900*v, color='#8A1C2B' if g else '#CFC6BF',
               ec='#57504D', lw=.7, alpha=.9, zorder=3)
axes[1].set_title(f'{TEAM} — {len(sh)} shots, sized by xG   ·   total xG {xg.sum():.2f}, goals {goal.sum()}')
plt.tight_layout(); plt.show()

g_scored, g_expected = int(goal.sum()), xg.sum()
print(f'{TEAM}: {len(ps)} passes, {len(sh)} shots, {g_expected:.2f} expected goals, {g_scored} scored.')
gap = g_scored - g_expected
verdict = ('outscored their chances' if gap > 0.5 else
           'underscored their chances' if gap < -0.5 else 'scored about what their chances were worth')
print(f'They {verdict} by {abs(gap):.2f} goals.')
print('\\nOver one match that difference is almost entirely luck. xG is a statement about the chances created,')
print('not about finishing ability — and one match is far too small a sample to separate the two.')


---
## Interpretation ✍️ **(graded)**

Pick **one** of your two figures and write 150 words a coach would act on. Then answer:

1. What tactical claim does your figure support, and what does it *not* support?
2. From Part A: one identity swap changed a distance-covered figure by the percentage you computed. What does that
   imply about a weekly report built from automatically tracked data?
3. `HIGH_SPEED_MS` is a threshold a vendor picks. Re-run Block 4 at 5.0 and at 5.8 m/s. How much does your
   squad's high-speed distance change? What does that mean for comparing two clubs' reports?

---

**Your answer:**

*(replace this line)*


---
## AI Use Log

Complete honestly — it is part of the mark. Write *none* if you used no AI tool.

| Tool | What I asked it | What it gave me | What I changed or rejected |
|---|---|---|---|
|  |  |  |  |


---
### Submitting

1. `File → Save a copy in Drive` · 2. `Share → Anyone with the link → Viewer` · 3. Paste the link into ODTÜClass.
